# PET-only 3D SUV volume with lesion contours — p-001 (68Ga-PSMA)

Builds the PET SUV volume straight from the TOF PET DICOM series (native grid, no CT
resampling) and renders it as a 3D isosurface with the three lesion masks contoured on top.
The CT series is only touched to decode the lesion RTSTRUCTs (they were contoured on CT) —
the CT volume itself is never loaded or displayed.

In [1]:
import shutil
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder, image_helper
from skimage.measure import marching_cubes

sys.path.append(str(Path('..').resolve() / 'scripts'))
from closest_anatomy import physical_points_to_indices

In [2]:

MTV_THRESHOLD_FRACTION = 0.41  # standard 41%-of-SUVmax MTV threshold


def dicom_time_to_seconds(time_str: str) -> float:
    time_str = str(time_str).split(".")[0]
    return int(time_str[0:2]) * 3600 + int(time_str[2:4]) * 60 + int(time_str[4:6])


def build_suv_volume(series_data) -> np.ndarray:
    """SUVbw volume in the same (Columns, Rows, slices) = (x, y, z) layout rt-utils masks use."""
    first = series_data[0]
    assert first.Units == "BQML", f"Unexpected PET Units: {first.Units}"

    weight_g = float(first.PatientWeight) * 1000
    rp = first.RadiopharmaceuticalInformationSequence[0]
    injected_dose_bq = float(rp.RadionuclideTotalDose)
    half_life_s = float(rp.RadionuclideHalfLife)
    injection_s = dicom_time_to_seconds(rp.RadiopharmaceuticalStartTime)
    reference_s = dicom_time_to_seconds(first.SeriesTime)
    decay_s = reference_s - injection_s
    if decay_s < 0:
        decay_s += 24 * 3600
    decayed_dose_bq = injected_dose_bq * (0.5 ** (decay_s / half_life_s))

    slices = []
    for s in series_data:
        raw = s.pixel_array.T.astype(np.float64)  # -> (Columns, Rows) = (x, y)
        slope = float(getattr(s, "RescaleSlope", 1.0))
        intercept = float(getattr(s, "RescaleIntercept", 0.0))
        slices.append(raw * slope + intercept)  # Bq/mL
    activity_bqml = np.stack(slices, axis=-1)  # (x, y, z)

    return activity_bqml * weight_g / decayed_dose_bq


def pet_space_to_sitk_image(array_xyz: np.ndarray, pixel_to_patient: np.ndarray) -> sitk.Image:
    """Wrap a (x, y, z) array (rt-utils layout) into a geometrically correct sitk.Image."""
    origin = pixel_to_patient[:3, 3].astype(np.float64)
    cols = [pixel_to_patient[:3, i].astype(np.float64) for i in range(3)]
    spacing = tuple(float(np.linalg.norm(c)) for c in cols)
    direction = np.column_stack([c / s for c, s in zip(cols, spacing)])

    array_numeric = array_xyz.astype(np.uint8) if array_xyz.dtype == bool else array_xyz
    img = sitk.GetImageFromArray(np.transpose(array_numeric, (2, 1, 0)))
    img.SetOrigin(tuple(float(v) for v in origin))
    img.SetSpacing(spacing)
    img.SetDirection(tuple(float(v) for v in direction.flatten()))
    return img

def voxel_to_physical(verts_ijk: np.ndarray, affine: np.ndarray) -> np.ndarray:
    homogeneous = np.concatenate([verts_ijk, np.ones((len(verts_ijk), 1))], axis=1)
    return (affine @ homogeneous.T).T[:, :3]

## 1. Load the PET series and build the SUV volume

Both DICOM exports for this patient have every slice duplicated on disk (same
`SOPInstanceUID`, two filenames). De-duplicate before handing folders to `rt_utils`/`SimpleITK`,
otherwise slice geometry gets corrupted.

In [3]:
PATIENT_DIR = Path('../data/psma/p-001')
OUTPUT_DIR = Path('../outputs/psma/p-001')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ct_dir = PATIENT_DIR / 'CT_WB_2_0_B26F_3_2MM_0004'
pet_dir = PATIENT_DIR / 'TOF_PET_WB_CORRECTED_0003'

lesion_rtstruct_files = {
    'anterior_prostate': next((PATIENT_DIR / 'ANTERIOR_PROSTATE_0001').glob('*.IMA')),
    'l_posterior_prostate': next((PATIENT_DIR / 'L_POSTERIOR_PROSTATE_0001').glob('*.IMA')),
    'r_posterior_prostate': next((PATIENT_DIR / 'R_POSTERIOR_PROSTATE_0001').glob('*.IMA')),
}


def dedup_dicom_dir(src_dir: Path, dst_dir: Path) -> Path:
    """Copy one file per unique SOPInstanceUID from src_dir into dst_dir."""
    dst_dir.mkdir(parents=True, exist_ok=True)
    seen = set()
    for f in sorted(src_dir.glob('*.IMA')):
        uid = pydicom.dcmread(f, stop_before_pixels=True).SOPInstanceUID
        if uid in seen:
            continue
        seen.add(uid)
        dst = dst_dir / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
    return dst_dir


ct_dedup_dir = dedup_dicom_dir(ct_dir, OUTPUT_DIR / 'ct_dedup')
pet_dedup_dir = dedup_dicom_dir(pet_dir, OUTPUT_DIR / 'pet_dedup')
print(f'CT unique slices: {len(list(ct_dedup_dir.glob("*.IMA")))}   PET unique slices: {len(list(pet_dedup_dir.glob("*.IMA")))}')

CT unique slices: 967   PET unique slices: 323


In [4]:
pet_series_data = image_helper.load_sorted_image_series(str(pet_dedup_dir))
suv_volume = build_suv_volume(pet_series_data)  # (x, y, z), native PET grid — no CT involved
pet_pixel_to_patient = image_helper.get_pixel_to_patient_transformation_matrix(pet_series_data)

print('PET SUV volume shape (x, y, z):', suv_volume.shape)
print(f'SUV min/mean/max: {suv_volume.min():.3f} / {suv_volume.mean():.3f} / {suv_volume.max():.3f}')

PET SUV volume shape (x, y, z): (200, 200, 323)
SUV min/mean/max: 0.000 / 0.093 / 97.751


## 2. Map each lesion into PET voxel space

The three lesion RTSTRUCTs were contoured on the **CT** grid. To contour them on the PET
volume without ever loading a CT image array, turn each lesion's CT voxel indices into
patient-space millimeters (`pixel_to_patient` from the CT series, decoded only to read the
RTSTRUCT geometry) and re-project those points into PET voxel indices with the PET
`pixel_to_patient` matrix above. A lesion's mask, expressed as a boolean array on the native
PET grid, is what gets contoured.

In [5]:
pet_shape_xyz = np.array(suv_volume.shape)
lesion_masks_pet = {}

for lesion_name, rtstruct_path in lesion_rtstruct_files.items():
    rtstruct = RTStructBuilder.create_from(dicom_series_path=str(ct_dedup_dir), rt_struct_path=str(rtstruct_path))
    roi_names = rtstruct.get_roi_names()
    masks = [rtstruct.get_roi_mask_by_name(r) for r in roi_names]
    combined_ct_mask = np.logical_or.reduce(masks)  # CT grid, (x, y, z)

    ct_pixel_to_patient = image_helper.get_pixel_to_patient_transformation_matrix(rtstruct.series_data)
    voxel_idx = np.argwhere(combined_ct_mask).astype(float)
    physical_pts = image_helper.apply_transformation_to_3d_points(voxel_idx, ct_pixel_to_patient)

    pet_idx = np.rint(physical_points_to_indices(physical_pts, pet_pixel_to_patient)).astype(int)
    in_bounds = np.all((pet_idx >= 0) & (pet_idx < pet_shape_xyz), axis=1)

    mask_pet = np.zeros(pet_shape_xyz, dtype=bool)
    ix, iy, iz = pet_idx[in_bounds, 0], pet_idx[in_bounds, 1], pet_idx[in_bounds, 2]
    mask_pet[ix, iy, iz] = True
    lesion_masks_pet[lesion_name] = mask_pet

    print(f'{lesion_name}: {combined_ct_mask.sum()} CT voxels -> {mask_pet.sum()} unique PET voxels '
          f'(SUVmax {suv_volume[mask_pet].max():.2f})')

anterior_prostate: 7620 CT voxels -> 229 unique PET voxels (SUVmax 21.30)


l_posterior_prostate: 32523 CT voxels -> 927 unique PET voxels (SUVmax 21.30)


r_posterior_prostate: 30335 CT voxels -> 847 unique PET voxels (SUVmax 21.30)


## 3. 3D render — PET isosurface + lesion contours

Marching cubes runs directly on the native PET grid (200×200×323, spacing ≈4.07×4.07×3.0 mm —
mild anisotropy, no isotropic resampling needed at `step_size=1`). Vertices are taken straight
from voxel index space and transformed to patient mm with `voxel_to_physical` using the PET
`pixel_to_patient` affine, so no CT geometry is involved anywhere in this render.

Whole-body 68Ga-PSMA activity is dominated by renal/bladder excretion (SUV up to ~98 here),
so the translucent "hot region" surface below is mostly urinary tract — the three lesion
meshes are drawn solid and labeled so they stand out against it.

In [6]:
def mesh_from_volume(volume_xyz: np.ndarray, level: float, affine: np.ndarray, step_size: int = 1):
    """Marching cubes over the whole array (e.g. the PET 'hot region')."""
    verts_ijk, faces, _, _ = marching_cubes(volume_xyz, level=level, step_size=step_size)
    return voxel_to_physical(verts_ijk, affine), faces


def mesh_from_local_mask(mask_xyz: np.ndarray, affine: np.ndarray, padding: int = 3):
    """Marching cubes cropped to the mask's bounding box. Returns None if empty/too small."""
    coords = np.argwhere(mask_xyz)
    if len(coords) == 0:
        return None
    lo = np.maximum(coords.min(axis=0) - padding, 0)
    hi = np.minimum(coords.max(axis=0) + padding + 1, np.array(mask_xyz.shape))
    crop = mask_xyz[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]].astype(np.float32)

    if crop.shape[0] < 2 or crop.shape[1] < 2 or crop.shape[2] < 2 or crop.max() == crop.min():
        return None
    try:
        verts_ijk, faces, _, _ = marching_cubes(crop, level=0.5)
    except (ValueError, RuntimeError):
        return None
    verts_ijk = verts_ijk + lo
    return voxel_to_physical(verts_ijk, affine), faces


def mesh_trace(mesh, color: str, opacity: float, name: str):
    verts, faces = mesh
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color, opacity=opacity, name=name, showlegend=True,
    )


def plot_3d_scene(title, pet_mesh=None, highlights=None):
    """highlights: list of (mesh, color, name, centroid_mm)."""
    traces = []
    if pet_mesh is not None:
        traces.append(mesh_trace(pet_mesh, 'orange', 0.2, 'PET hot region (SUV>=2.5)'))

    annotation_points = {'x': [], 'y': [], 'z': [], 'text': []}
    for mesh, color, name, centroid_mm in (highlights or []):
        if mesh is None:
            print(f'  [warn] could not build mesh for {name}, skipping')
            continue
        traces.append(mesh_trace(mesh, color, 0.9, name))
        annotation_points['x'].append(centroid_mm[0])
        annotation_points['y'].append(centroid_mm[1])
        annotation_points['z'].append(centroid_mm[2])
        annotation_points['text'].append(name)

    if annotation_points['x']:
        traces.append(go.Scatter3d(
            x=annotation_points['x'], y=annotation_points['y'], z=annotation_points['z'],
            mode='markers+text', text=annotation_points['text'], textposition='top center',
            marker=dict(size=3, color='black'), name='labels', showlegend=False,
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data', xaxis_title='x (mm)', yaxis_title='y (mm)', zaxis_title='z (mm)'),
        width=950, height=750, legend=dict(itemsizing='constant'),
    )
    return fig

In [7]:
LESION_COLORS = {
    'anterior_prostate': 'crimson',
    'l_posterior_prostate': 'royalblue',
    'r_posterior_prostate': 'seagreen',
}

pet_hot_mesh = mesh_from_volume(suv_volume, level=2.5, affine=pet_pixel_to_patient, step_size=1)

highlights = []
for name, mask in lesion_masks_pet.items():
    mesh = mesh_from_local_mask(mask, pet_pixel_to_patient)
    centroid_mm = voxel_to_physical(np.argwhere(mask).mean(axis=0, keepdims=True), pet_pixel_to_patient)[0]
    highlights.append((mesh, LESION_COLORS[name], name, centroid_mm))

fig = plot_3d_scene(
    title='PET SUV volume — p-001 (68Ga-PSMA), lesions contoured (CT never loaded)',
    pet_mesh=pet_hot_mesh,
    highlights=highlights,
)
fig.show()

In [8]:
def create_coronal_mip_with_selection():
  """Create a coronal MIP with interactive bounding box selection."""
  # Compute coronal MIP (max along x-axis)
  coronal_mip = np.max(suv_volume, axis=0)  # (y, z)
  
  fig = go.Figure()
  
  # Add MIP as heatmap
  fig.add_trace(go.Heatmap(
    z=coronal_mip,
    colorscale='Viridis',
    name='Coronal MIP',
    hovertemplate='y: %{x}<br>z: %{y}<br>SUV: %{z:.2f}<extra></extra>',
  ))
  
  # Add rectangle shape for selection (initially hidden)
  fig.add_shape(
    type='rect',
    x0=0, y0=0, x1=10, y1=10,
    line=dict(color='red', width=2),
    name='Selection Box',
    visible=False,
  )
  
  fig.update_layout(
    title='Coronal MIP - Drag to select region',
    xaxis_title='y (voxels)',
    yaxis_title='z (voxels)',
    width=900,
    height=700,
    dragmode='drawrect',
    newshape_line_color='red',
  )
  
  fig.show()
  
  return coronal_mip

coronal_mip = create_coronal_mip_with_selection()

In [9]:
# Overlay lesion masks on the coronal MIP (project each 3D mask onto the y-z plane)
fig = go.Figure()

fig.add_trace(go.Heatmap(
  z=coronal_mip,
  colorscale='Viridis',
  name='Coronal MIP',
  hovertemplate='y: %{x}<br>z: %{y}<br>SUV: %{z:.2f}<extra></extra>',
))

for lesion_name, mask_pet in lesion_masks_pet.items():
  mask_yz = mask_pet.any(axis=0).astype(np.uint8)  # project x -> y-z
  fig.add_trace(go.Contour(
    z=mask_yz,
    x=np.arange(mask_yz.shape[1]),  # z
    y=np.arange(mask_yz.shape[0]),  # y
    contours=dict(start=0.5, end=0.5, size=1, coloring='none'),
    line=dict(color=LESION_COLORS.get(lesion_name, 'red'), width=3),
    showscale=False,
    name=lesion_name,
    hoverinfo='skip',
  ))

fig.update_layout(
  title='Coronal MIP with lesion mask overlays',
  xaxis_title='z (voxels)',
  yaxis_title='y (voxels)',
  width=900,
  height=700,
)

fig.show()